In [1]:
# Cell 0: Initialize and Load Data
import json
from pathlib import Path
from groq import Groq
from dotenv import load_dotenv
import os

# Load environment
load_dotenv()

# Set up paths
NOTEBOOK_DIR = Path.cwd()
BASE_DIR = NOTEBOOK_DIR.parent
DATA_DIR = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Initialize Groq client
api_key = os.getenv('GROQ_API_KEY')
client = Groq(api_key=api_key)
print('✓ Groq client initialized')

# Load queries by ambiguity level
with open(DATA_DIR / 'queries_input_updated.json', 'r') as f:
    queries_by_level = json.load(f)

level_1 = queries_by_level['level_1_clear']
level_2 = queries_by_level['level_2_medium']
level_3 = queries_by_level['level_3_vague']

print(f'✓ Loaded {len(level_1)} Level 1 queries')
print(f'✓ Loaded {len(level_2)} Level 2 queries')
print(f'✓ Loaded {len(level_3)} Level 3 queries')

✓ Groq client initialized
✓ Loaded 6 Level 1 queries
✓ Loaded 5 Level 2 queries
✓ Loaded 7 Level 3 queries


In [ ]:
# Cell 1: Define Interpreter Function
INTERPRETER_PROMPT = """Analyze this query and extract the user's analytical intent.

Query: {input_text}

Respond in exactly this format (3 lines):
ACTION: [one of: PREDICT, COMPARE, CORRELATE, SUMMARIZE, EXPLAIN, FILTER, JOIN, or ERROR]
CONFIDENCE: [0.0 to 1.0]
REASONING: [brief explanation in one sentence]

Action Definitions:
- PREDICT: Forecast future outcomes, trends, or forecasts
- COMPARE: Contrast multiple entities, groups, values, or practices
- CORRELATE: Find relationships, associations, or dependencies between variables
- SUMMARIZE: Aggregate data, describe patterns, or highlight interesting/notable findings
- EXPLAIN: Understand WHY something happens or HOW processes work
- FILTER: Find specific records, rows, or items matching criteria or conditions
- JOIN: Link, combine, or merge multiple datasets together by matching records

CRITICAL DISTINCTIONS (Common Confusions):
- "Link/join/combine datasets" → JOIN (NOT CORRELATE)
  "Link vaccine batches with adverse events" = JOIN (linking two tables)
- "Why/how does X work?" → EXPLAIN
  "Why do regions differ?" = EXPLAIN (understanding causation)
- "Tell me something interesting/notable" → SUMMARIZE (find patterns)
  NOT EXPLAIN - user wants interesting patterns, not explanations
- "Anything wrong/problems/bad batches?" → FILTER (find problematic records)
  NOT EXPLAIN - user wants to identify issues, not understand them
- "Understand everything about X" → SUMMARIZE (overview of all data/patterns)

Think step-by-step:
1. Identify explicit keywords from the Action Definitions above
2. For ambiguous cases, use CRITICAL DISTINCTIONS to disambiguate
3. If none match, check for implicit intent structure
4. Set confidence: 0.9+ for explicit keywords, 0.7-0.8 for implicit, <0.7 for vague
5. Return ERROR only if query is nonsensical or off-domain
"""

def extract_intent(input_text, client):
    import time
    time.sleep(0.2)  # Rate limiting
    
    prompt = INTERPRETER_PROMPT.format(input_text=input_text)
    response = client.chat.completions.create(
        model='groq/compound',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.1,
        max_tokens=100
    )
    
    response_text = response.choices[0].message.content.strip()
    lines = response_text.split('\n')
    
    action = 'ERROR'
    confidence = 0.0
    reasoning = ''
    
    for line in lines:
        if line.startswith('ACTION:'):
            action = line.replace('ACTION:', '').strip()
        elif line.startswith('CONFIDENCE:'):
            try:
                confidence = float(line.replace('CONFIDENCE:', '').strip())
            except:
                confidence = 0.0
        elif line.startswith('REASONING:'):
            reasoning = line.replace('REASONING:', '').strip()
    
    return {
        'q': input_text,
        'intended_action': action,
        'confidence': confidence,
        'reasoning': reasoning
    }

print('✓ extract_intent function defined')

✓ extract_intent function defined


In [3]:
# Cell: Test all levels with LONGER batch delays

import time

print('\n' + '='*70)
print('TESTING ALL LEVELS WITH LONGER DELAYS (TPM RESET)')
print('='*70)

all_results = []

# Level 1
print('\nLEVEL 1: CLEAR QUERIES')
level_1_results = []
for i, q in enumerate(level_1, 1):
    result = extract_intent(q, client)
    level_1_results.append(result)
    print(f'Q{i}: {result["intended_action"]:12s} (conf: {result["confidence"]:.2f})')
    time.sleep(0.5)

all_results.extend(level_1_results)
print('✓ Level 1 done. Waiting 65s for TPM reset before Level 2...')
time.sleep(65)  # Wait for TPM counter to reset

# Level 2
print('\nLEVEL 2: MEDIUM AMBIGUOUS QUERIES')
level_2_results = []
for i, q in enumerate(level_2, 1):
    result = extract_intent(q, client)
    level_2_results.append(result)
    print(f'Q{i}: {result["intended_action"]:12s} (conf: {result["confidence"]:.2f})')
    time.sleep(0.5)

all_results.extend(level_2_results)
print('✓ Level 2 done. Waiting 65s for TPM reset before Level 3...')
time.sleep(65)

# Level 3
print('\nLEVEL 3: VAGUE QUERIES')
level_3_results = []
for i, q in enumerate(level_3, 1):
    result = extract_intent(q, client)
    level_3_results.append(result)
    print(f'Q{i}: {result["intended_action"]:12s} (conf: {result["confidence"]:.2f})')
    time.sleep(0.5)

all_results.extend(level_3_results)

# Save results
output = {
    'metadata': {
        'timestamp': '2026-09-16',
        'total_queries': len(all_results),
        'level_1_count': len(level_1_results),
        'level_2_count': len(level_2_results),
        'level_3_count': len(level_3_results),
        'interpreter_version': '2.1-improved-prompt'
    },
    'results': [
        {
            'query': r['q'],
            'intended_action': r['intended_action'],
            'confidence': r['confidence'],
            'reasoning': r['reasoning']
        }
        for r in all_results
    ]
}

output_path = RESULTS_DIR / 'interpreter_output.json'
with open(output_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'\n✓ Saved all {len(all_results)} results to interpreter_output.json!')
print(f'  - Level 1: {len(level_1_results)} queries')
print(f'  - Level 2: {len(level_2_results)} queries')
print(f'  - Level 3: {len(level_3_results)} queries')


TESTING ALL LEVELS WITH LONGER DELAYS (TPM RESET)

LEVEL 1: CLEAR QUERIES
Q1: PREDICT      (conf: 0.90)
Q2: COMPARE      (conf: 0.95)
Q3: CORRELATE    (conf: 0.95)
Q4: SUMMARIZE    (conf: 0.95)
Q5: FILTER       (conf: 0.95)
Q6: JOIN         (conf: 0.95)
✓ Level 1 done. Waiting 65s for TPM reset before Level 2...

LEVEL 2: MEDIUM AMBIGUOUS QUERIES
Q1: EXPLAIN      (conf: 0.95)
Q2: CORRELATE    (conf: 0.95)
Q3: COMPARE      (conf: 0.95)
Q4: PREDICT      (conf: 0.95)
Q5: CORRELATE    (conf: 0.95)
✓ Level 2 done. Waiting 65s for TPM reset before Level 3...

LEVEL 3: VAGUE QUERIES
Q1: COMPARE      (conf: 0.95)
Q2: SUMMARIZE    (conf: 0.95)
Q3: SUMMARIZE    (conf: 0.90)
Q4: SUMMARIZE    (conf: 0.80)
Q5: FILTER       (conf: 0.70)
Q6: EXPLAIN      (conf: 0.80)
Q7: FILTER       (conf: 0.90)

✓ Saved all 18 results to interpreter_output.json!
  - Level 1: 6 queries
  - Level 2: 5 queries
  - Level 3: 7 queries
